# Week 4 — LangGraph exercise (community contribution)

**Author:** Emmanuel  

## Workflow (`workflow_graph.py`)

1. **classify_small_talk** — routes chit-chat vs substantive requests.  
2. **handle_small_talk** — short reply, then **END**.  
3. **clarifying_agent** — if the request is too vague, asks the user and **END**; else continues.  
4. **planning_agent** — structured research plan (goal + ordered steps) before searching.  
5. **search_agent** — **`internet_search`** (DuckDuckGo) + LLM synthesis (prompts include the plan).  
6. **evaluator** — structured LLM gate; on failure loops back to **search_agent**.

Run the course virtualenv (`langgraph`, `langchain_core`, `gradio`, `ddgs`). Use the **Gradio** cell below for an interactive chat (checkpointed by conversation).

In [ ]:
import uuid

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

from workflow_graph import graph

# With MemorySaver, every invoke needs a thread_id (use a new UUID per isolated run).
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Example: substantive question (clarifier → search → tool → search → evaluator → END)
out = graph.invoke(
    {
        "messages": [HumanMessage(content="Summarize recent news about electric vehicles in Europe.")],
        "is_small_talk": None,
        "needs_clarification": None,
        "research_plan": None,
        "evaluation_passed": None,
    },
    config=config,
)
for m in out["messages"]:
    print(type(m).__name__, "-", (m.content or str(getattr(m, "tool_calls", "")))[:200])

In [ ]:
import uuid

import gradio as gr
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage

load_dotenv(override=True)

from workflow_graph import graph


def _gradio_history_to_messages(history: list) -> list:
    """Rebuild LangChain messages from the Chatbot transcript (source of truth for memory)."""
    out = []
    for msg in history:
        if not isinstance(msg, dict):
            continue
        role = msg.get("role")
        content = msg.get("content", "")
        if isinstance(content, list):
            content = " ".join(str(x) for x in content)
        else:
            content = str(content)
        if role == "user":
            out.append(HumanMessage(content=content))
        elif role == "assistant":
            out.append(AIMessage(content=content))
    return out


def last_assistant_reply(messages: list) -> str:
    for m in reversed(messages):
        if isinstance(m, AIMessage):
            text = (m.content or "").strip()
            if text:
                return text
    return "(No assistant text in response.)"


PLAN_PLACEHOLDER = "*Research plan steps appear here for substantive questions (right column).*"


def format_plan_markdown(plan_text: str | None) -> str:
    if not (plan_text or "").strip():
        return "*No plan this turn (small talk, clarification, or direct reply).*"
    body = plan_text.strip().replace("\n", "\n\n")
    return f"## Research plan\n\n{body}"


def chat_fn(user_message: str, history: list):
    """Send full visible conversation each time; fresh thread_id avoids checkpoint duplication."""
    text = user_message.strip()
    messages = _gradio_history_to_messages(history)
    messages.append(HumanMessage(content=text))
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    state = {
        "messages": messages,
        "is_small_talk": None,
        "needs_clarification": None,
        "research_plan": None,
        "evaluation_passed": None,
    }
    result = graph.invoke(state, config=config)
    reply = last_assistant_reply(result["messages"])
    history = history + [
        {"role": "user", "content": text},
        {"role": "assistant", "content": reply},
    ]
    plan_md = format_plan_markdown(result.get("research_plan"))
    return history, "", plan_md


def reset_fn():
    return [], "", PLAN_PLACEHOLDER


with gr.Blocks(theme=gr.themes.Default(primary_hue="slate")) as demo:
    gr.Markdown("## LangGraph workflow chat")
    gr.Markdown(
        "Substantive questions run search + evaluation. **New conversation** clears the chat. "
        "Memory comes from the chat transcript on each send (works reliably with Gradio)."
    )
    with gr.Row(equal_height=True):
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Conversation", height=420, type="messages")
        with gr.Column(scale=2):
            gr.Markdown("### Planning")
            plan_panel = gr.Markdown(value=PLAN_PLACEHOLDER)
    user_in = gr.Textbox(placeholder="Your message…", show_label=False, lines=2)
    with gr.Row():
        send = gr.Button("Send", variant="primary")
        reset = gr.Button("New conversation")

    send.click(chat_fn, [user_in, chatbot], [chatbot, user_in, plan_panel])
    user_in.submit(chat_fn, [user_in, chatbot], [chatbot, user_in, plan_panel])
    reset.click(reset_fn, None, [chatbot, user_in, plan_panel])

demo.launch(share=False)

## Optional: visualize

Uncomment when dependencies are installed.

```python
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))
```

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))